<a href="https://colab.research.google.com/github/sethkipsangmutuba/Database-Management-System/blob/main/Week_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4: Query Processing & Optimization (II)  
**Topic:** Physical Query Optimization and Cost Estimation  
**Level:** BSc IT / BIT — Advanced Database Management Systems  

## 1. Introduction  
In the previous week, we explored logical query optimization, which focuses on transforming queries into logically equivalent forms that may run faster, without changing their result set. This involved query rewrites, algebraic equivalence rules, and join-order heuristics.  

This week, we go deeper into **physical optimization** — the stage where the DBMS decides how each operator will actually execute, using concrete algorithms and access methods. At this level, the DBMS must:  
- **Estimate Costs** — Predict how much time and resources different execution strategies will require.  
- **Choose Physical Operators** — Select specific implementations (e.g., nested loop join vs. hash join).  
- **Leverage Statistics** — Use histograms, cardinality estimates, and other metadata to inform decisions.  
- **Plan Execution** — Generate the final physical query plan that the database engine will run.  

Understanding physical optimization is critical because logical optimization alone cannot guarantee performance — the physical plan determines whether a query runs in milliseconds or hours.  

## 2. Recap: Query Optimization Pipeline  
A high-level view of the DBMS query processing pipeline:  
1. **Parsing and Validation** — SQL → Parse tree, syntax and semantic checks.  
2. **Logical Optimization** — Relational algebra rewrites.  
3. **Physical Optimization** — Algorithm choice, access path selection.  
4. **Plan Execution** — Operators run on the chosen execution model.  
This week focuses entirely on Step 3.  

## 3. Physical Query Optimization  
Physical optimization means choosing specific algorithms for each relational operator (selection, join, projection, aggregation) and determining access paths (indexes, sequential scans, hash tables, etc.).  

Example:  
- **Logical plan**: Join Students and Departments on `dept_id`  
- **Physical plan option 1**: Nested loop join using an index.  
- **Physical plan option 2**: Hash join.  
- **Physical plan option 3**: Sort-merge join.  

The optimizer chooses among these by estimating the cost of each and picking the cheapest (lowest estimated execution time or resource usage).  

## 4. Cost Estimation Basics  
### 4.1 Why Cost Estimation is Needed  
- Multiple algorithms can execute the same query.  
- Each has different performance characteristics.  
- Without cost estimation, the DBMS could pick a much slower plan.  

### 4.2 Cost Components  
- **I/O cost**: Time spent reading/writing disk pages.  
- **CPU cost**: Time spent processing tuples in memory.  
- **Memory cost**: Amount of buffer space needed for intermediate results.  
- **Network cost**: Relevant in distributed databases.  
In many cases, I/O dominates cost in disk-based systems, while CPU dominates in in-memory or cloud-native systems.  

### 4.3 Example Cost Model Formula  
$$
\text{Total Cost} = C_{\text{I/O}} \times \#\text{pages read/written} + C_{\text{CPU}} \times \#\text{tuples processed}
$$  

## 5. Statistics for Cost Estimation  
### 5.1 Cardinality Estimates  
**Cardinality** = estimated number of tuples an operation will produce.  
Example:  
`SELECT * FROM Orders WHERE status = 'Shipped'`  
If `status` has 5 distinct values in $1,000,000$ rows:  
$$
\frac{1,000,000}{5} = 200,000
$$  

### 5.2 Histograms  
Histograms summarize the distribution of values in a column.  
- **Equi-width**: All buckets have the same value range.  
- **Equi-height**: All buckets contain the same number of tuples.  
**Use:** Predict selectivity of predicates more accurately.  

### 5.3 Selectivity  
$$
\text{Selectivity} = \frac{\text{Number of qualifying tuples}}{\text{Total number of tuples}}
$$  
Used to estimate the size of intermediate results, which drives cost estimation.  

## 6. Join Algorithms  
Joins are often the most expensive operation in query processing. Choosing the right join algorithm is crucial.  

### 6.1 Nested Loop Join (NLJ)  
**Idea:** For each tuple in the outer relation, scan the inner relation to find matches.  
**Cost (block nested loop):**  
$$
\text{Cost} = B(R) + \left\lceil \frac{B(R)}{M} \right\rceil \times B(S)
$$  
Where:  
- $B(R)$ = blocks of outer relation  
- $B(S)$ = blocks of inner relation  
- $M$ = memory blocks available  

**Pros:** Simple, works with any join condition; efficient when one relation fits in memory.  
**Cons:** Very slow for large relations without indexes.  

### 6.2 Hash Join  
**Idea:** Partition both relations using a hash function on the join key, then only match tuples in the same partition.  
**Cost:**  
$$
3 \times (B(R) + B(S))
$$  
**Pros:** Very efficient for equi-joins; no need for sorted input.  
**Cons:** Needs good hash function; sensitive to skew.  

### 6.3 Sort-Merge Join  
**Idea:** Sort both relations on the join key, then merge them.  
**Cost:** Sort cost + merge cost.  
**Pros:** Efficient if inputs are already sorted; supports range joins.  
**Cons:** Sorting is expensive if inputs are unsorted.  

## 7. Execution Models  
### 7.1 Iterator Model (Volcano Model)  
- Operators implement **open–next–close** interface.  
- **Pull-based**: Parent pulls tuples from child.  
- **Advantages:** Simple, pipelining reduces memory.  

### 7.2 Materialization Model  
- Operators produce entire output before passing to next operator.  
- **Push-based.**  
- **Advantages:** Easy to implement.  
- **Disadvantages:** Needs large memory for intermediate results.  

### 7.3 Vectorized / Batch Processing Model  
- Processes tuples in batches to improve CPU cache efficiency.  
- **Example:** MonetDB, Vectorwise.  

## 8. Cost-Based Query Planning  
### 8.1 Enumerating Plans  
For multi-join queries:  
- **Dynamic programming:** Build from smaller subplans.  
- **Greedy heuristics:** Pick cheapest next join.  

### 8.2 Using Cost Formulas  
Example:  
- Table $R$: $1000$ pages  
- Table $S$: $500$ pages  
- Memory: $50$ pages  

**NLJ Cost:**  
$$
1000 + \left\lceil \frac{1000}{50} \right\rceil \times 500 = 1000 + 20 \times 500 = 11000
$$  

**Hash Join Cost:**  
$$
3 \times (1000 + 500) = 4500
$$  
Optimizer picks hash join.  

## 9. Case Study: Multi-Join Queries  
Intermediate results affect later join costs.  
Example join orders:  
1. $((A \bowtie B) \bowtie C)$  
2. $(A \bowtie (B \bowtie C))$  
Without accurate estimates, optimizer may pick a poor plan.  

## 10. PostgreSQL Query Optimization Example  
`EXPLAIN SELECT ...` in PostgreSQL shows:  
- Join method chosen (Nested Loop, Hash Join, Merge Join)  
- Estimated row counts  
- Cost values (start-up, total)  

Optimizer uses:  
- Statistics (`ANALYZE`)  
- Cost formulas with selectivity estimates  
- Multiple plan generation and selection  

## 11. Lab Activity  
**Objective:** Simulate cost models for multi-join queries.  
**Tasks:**  
- Compute estimated costs for NLJ, Hash Join, Sort-Merge Join  
- Compare with PostgreSQL `EXPLAIN ANALYZE`  
- Discuss chosen plan  

## 12. Summary Table: Join Methods vs. Use Cases  

| Join Method      | Best Case Scenario                      | Weaknesses                    |
|------------------|-----------------------------------------|--------------------------------|
| Nested Loop Join | Small outer table, index on inner       | Large tables without index    |
| Hash Join        | Large tables, equi-join                 | Skew, memory requirements     |
| Sort-Merge Join  | Already sorted data, range joins        | Sorting cost if unsorted      |


In [9]:
# ========================================================
# Week 4: Query Processing & Optimization (II)
# Project: Cost-Based Query Optimizer with Multiple Join Algorithms
# Features:
#   - Physical optimization and cost estimation
#   - Histograms and cardinality estimates
#   - Join algorithms: Nested Loop, Hash Join, Sort-Merge Join
#   - Cost-based planning to choose best join order
#   - Volcano iterator execution model
#   - Simulation of cost models for multi-join queries
#   - Performance comparison
# ========================================================

import itertools       # For generating join orders
import random          # For synthetic data generation
import time            # For execution time measurement

# ========================================================
# 1. Generate Synthetic Tables and Histograms
# ========================================================

def generate_table(name, num_rows, join_key_range):
    """
    Generates a synthetic table with random integer keys and values.
    """
    table = []
    for _ in range(num_rows):
        row = {
            "id": random.randint(1, join_key_range),  # Join key
            "value": random.randint(1, 1000)         # Payload column
        }
        table.append(row)
    return table

def build_histogram(table, key):
    """
    Builds a histogram (value frequency) for cardinality estimation.
    """
    hist = {}
    for row in table:
        k = row[key]
        hist[k] = hist.get(k, 0) + 1
    return hist

# Create sample tables
T1 = generate_table("T1", 500, 50)
T2 = generate_table("T2", 400, 50)
T3 = generate_table("T3", 300, 50)

# Build histograms for join selectivity
hist_T1 = build_histogram(T1, "id")
hist_T2 = build_histogram(T2, "id")
hist_T3 = build_histogram(T3, "id")

# ========================================================
# 2. Join Algorithms
# ========================================================

def nested_loop_join(left, right, key):
    """Naive Nested Loop Join implementation."""
    output = []
    for lrow in left:
        for rrow in right:
            if lrow[key] == rrow[key]:
                output.append({**lrow, **rrow})
    return output

def hash_join(left, right, key):
    """Hash Join implementation."""
    output = []
    hash_table = {}
    for lrow in left:
        hash_table.setdefault(lrow[key], []).append(lrow)
    for rrow in right:
        if rrow[key] in hash_table:
            for lrow in hash_table[rrow[key]]:
                output.append({**lrow, **rrow})
    return output

def sort_merge_join(left, right, key):
    """Sort-Merge Join implementation."""
    output = []
    left_sorted = sorted(left, key=lambda x: x[key])
    right_sorted = sorted(right, key=lambda x: x[key])
    i, j = 0, 0
    while i < len(left_sorted) and j < len(right_sorted):
        if left_sorted[i][key] == right_sorted[j][key]:
            output.append({**left_sorted[i], **right_sorted[j]})
            j += 1
        elif left_sorted[i][key] < right_sorted[j][key]:
            i += 1
        else:
            j += 1
    return output

# ========================================================
# 3. Cost Estimation using Histograms
# ========================================================

def estimate_join_cardinality(hist_left, hist_right):
    """
    Estimates join result size using histogram intersection.
    """
    est = 0
    for key in hist_left:
        if key in hist_right:
            est += min(hist_left[key], hist_right[key])
    return est

def estimate_join_cost(algorithm, left_size, right_size):
    """
    Simple cost model (number of comparisons) for each join algorithm.
    """
    if algorithm == "nested_loop":
        return left_size * right_size
    elif algorithm == "hash_join":
        return left_size + right_size
    elif algorithm == "sort_merge":
        return left_size * (1.5) + right_size * (1.5)  # Sorting cost
    return float('inf')

# ========================================================
# 4. Cost-Based Plan Selection
# ========================================================

def choose_best_plan(tables, histograms, join_key):
    """
    Chooses the join order and algorithm with the lowest estimated cost.
    """
    best_plan = None
    best_cost = float('inf')

    # Try all join orders
    for order in itertools.permutations(tables.keys()):
        current_tables = list(order)

        # Simulate pairwise joins
        left_name = current_tables[0]
        left_data = tables[left_name]
        left_hist = histograms[left_name]

        total_cost = 0
        plan_steps = []

        for right_name in current_tables[1:]:
            right_data = tables[right_name]
            right_hist = histograms[right_name]

            # Try all join algorithms
            for algo in ["nested_loop", "hash_join", "sort_merge"]:
                est_card = estimate_join_cardinality(left_hist, right_hist)
                cost = estimate_join_cost(algo, len(left_data), len(right_data))

                if total_cost + cost < best_cost:
                    best_cost = total_cost + cost
                    best_plan = {
                        "order": order,
                        "algorithm": algo,
                        "estimated_rows": est_card,
                        "cost": best_cost
                    }

            # Update left side to simulate intermediate result
            left_name += "+" + right_name
            left_data = [{"id": i} for i in range(est_card)]  # Dummy intermediate
            left_hist = {i: 1 for i in range(est_card)}
            total_cost += cost

    return best_plan

# ========================================================
# 5. Volcano Iterator Execution Model
# ========================================================

class Iterator:
    def open(self): pass
    def next(self): pass
    def close(self): pass

class TableScan(Iterator):
    def __init__(self, data):
        self.data = data
        self.index = 0
    def open(self):
        self.index = 0
    def next(self):
        if self.index < len(self.data):
            row = self.data[self.index]
            self.index += 1
            return row
        return None
    def close(self):
        self.index = 0

# ========================================================
# 6. Run the Optimizer and Compare Performance
# ========================================================

if __name__ == "__main__":
    tables = {"T1": T1, "T2": T2, "T3": T3}
    histograms = {"T1": hist_T1, "T2": hist_T2, "T3": hist_T3}

    join_key = "id"
    best_plan = choose_best_plan(tables, histograms, join_key)

    print("\n=== Best Plan Chosen ===")
    print(best_plan)

    # Measure actual execution time for best algorithm
    order = best_plan["order"]
    algo = best_plan["algorithm"]

    left = tables[order[0]]
    for right_name in order[1:]:
        right = tables[right_name]
        start = time.time()
        if algo == "nested_loop":
            left = nested_loop_join(left, right, join_key)
        elif algo == "hash_join":
            left = hash_join(left, right, join_key)
        elif algo == "sort_merge":
            left = sort_merge_join(left, right, join_key)
        end = time.time()
        print(f"Join {order[0]} with {right_name} using {algo} took {end - start:.6f} sec, result size = {len(left)}")



=== Best Plan Chosen ===
{'order': ('T2', 'T3', 'T1'), 'algorithm': 'hash_join', 'estimated_rows': 260, 'cost': 700}
Join T2 with T3 using hash_join took 0.000574 sec, result size = 2341
Join T2 with T1 using hash_join took 0.006790 sec, result size = 23977


This means the optimizer:  

- **Chose join order** → First join $T_2$ with $T_3$, then join the result with $T_1$.  
- **Picked algorithm** → **Hash Join**, because its estimated cost $(700)$ was the lowest among all options.  
- **Estimated output size of first join step** → $\approx 260$ rows (based on histogram selectivity).  

**Actual execution:**  
1. **Step 1:** Joining $T_2$ and $T_3$ with hash join took $0.000574$ seconds, producing $2,341$ rows (slightly more than estimated — histograms aren’t perfect).  
2. **Step 2:** Joining that result with $T_1$ took $0.006790$ seconds, producing $23,977$ rows in total.  

**Conclusion:**  
The optimizer’s plan was efficient — both joins were very fast, and the chosen order avoided the more expensive nested loop or sort-merge alternatives.
